In [11]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datasets import WM_811K
wm811k = WM_811K(imsize=(64,64))



reading wafermaps


  0%|          | 0/12822 [00:00<?, ?it/s]

In [ ]:
ds_train = wm811k.dataset_multi_defect_fullstack().batch(16)
ds_validation = wm811k.dataset_multi_defect_fullstack().batch(16)

model_segmentation = tf.keras.models.load_model(os.path.join('..', 'models', 'multi_defect_segmentation.keras'))

model_segmentation.trainable = False 


class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        """
        kernel_size: 7x7 is the standard established by the CBAM paper. 
        It gives the convolution a wide enough receptive field to understand 
        the geometry around a pixel.
        """
        super(SpatialAttention, self).__init__(**kwargs)
        
        # The single convolution layer that creates the final mask
        self.conv = tf.keras.layers.Conv2D(
            filters=1, 
            kernel_size=kernel_size, 
            padding='same', 
            activation='sigmoid', 
            use_bias=False # Bias is unnecessary here
        )

    def call(self, inputs):
        # 1. Average Pooling across the channel axis (axis=-1)
        # Shape changes from (B, H, W, C) -> (B, H, W, 1)
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        
        # 2. Max Pooling across the channel axis
        # Shape changes from (B, H, W, C) -> (B, H, W, 1)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        
        # 3. Concatenate the two maps together
        # Shape becomes (B, H, W, 2)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        
        # 4. Pass through convolution and sigmoid to generate the mask
        # Shape becomes (B, H, W, 1), with values between 0.0 and 1.0
        attention_mask = self.conv(concat)
        
        # 5. Multiply the mask against the original inputs
        return inputs * attention_mask

    # Required so you can save and load your model later without errors
    def get_config(self):
        config = super(SpatialAttention, self).get_config()
        return config

model_classification = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(*wm811k.imsize,WM_811K.n_classes-1)),
                          
    # ---- Conv block 1 -------------------------------------------------
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    SpatialAttention(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Conv block 2 -------------------------------------------------
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    SpatialAttention(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Conv block 3 -------------------------------------------------
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    SpatialAttention(),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Flatten + Dense -----------------------------------------------
    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(WM_811K.n_classes-1, activation='sigmoid'),
#    tf.keras.layers.Dense(WM_811K.n_classes-1, activation='softmax')
    
    ])

model_classification.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001),
)

model_classification.summary()




Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 62, 62, 32)     │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 31, 31, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 6, 6, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 512)            │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_24 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 8)              │         4,104 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,726,248 (10.40 MB)

 Trainable params: 2,723,752 (10.39 MB)

 Non-trainable params: 2,496 (9.75 KB)

In [10]:
# stage 1: Train Classification Only

#freeze  segmentation model
model_segmentation.trainable = False


model_fullstack = tf.keras.models.Sequential([
    model_segmentation,
    model_classification 
])

model_fullstack.summary()

model_fullstack.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
    metrics=["accuracy"]
)

# Train Stage 1
print("Starting Stage 1: Training Classification Head Only...")
history_stage1 = model_fullstack.fit(
    x=ds_train,
    steps_per_epoch=128,
    validation_data=ds_validation,
    validation_steps=32,
    epochs=20, # Train just enough for the classification head to converge
    #callbacks=callbacks,
    verbose=1
)

# stage 2 Train the Entire Model

#  Unfreeze  segmentation model
model_segmentation.trainable = True

model_fullstack.compile(
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), # Reduced learning rate
    metrics=["accuracy"]
)

#  Train Stage 2
print("Starting Stage 2: Fine-tuning Full Model...")
history_stage2 = model_fullstack.fit(
    x=ds_train,
    steps_per_epoch=128,
    validation_data=ds_validation,
    validation_steps=32,
    epochs=50, 
    # callbacks=callbacks,
    verbose=1
)

def plot_training_curves(history_dict, out_path):
    epochs  = range(1, len(history_dict["loss"]) + 1)
    best_ep = history_dict["best_epoch"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for ax, metric in [(ax1, "accuracy"), (ax2, "loss")]:
        ax.plot(epochs, history_dict[metric],          label="Train",      linewidth=2)
        ax.plot(epochs, history_dict[f"val_{metric}"], label="Validation", linewidth=2, linestyle="--")
        ax.axvline(best_ep, color="gray", linestyle=":", label=f"Best epoch ({best_ep})")
        ax.set_title(metric.capitalize()); ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig(out_path, dpi=150); plt.show()
    print(f"Saved {out_path}")

history_dict = dict(history_stage1.history)
history_dict = {k: [float(v) for v in vals] for k, vals in history_stage1.history.items()}
history_dict["best_epoch"]        = int(np.argmin(history_stage1.history["val_loss"])) + 1
history_dict["best_val_accuracy"] = float(min(history_stage1.history["val_loss"]))

OUTPUT_DIR   = "./performance_plots"   # all performance plots written here

plot_training_curves(history_dict, out_path=os.path.join(OUTPUT_DIR, "training_curves.png"))

history_dict2 = dict(history_stage2.history)
history_dict2 = {k: [float(v) for v in vals] for k, vals in history_stage2.history.items()}
history_dict2["best_epoch"]        = int(np.argmin(history_stage2.history["val_loss"])) + 1
history_dict2["best_val_accuracy"] = float(min(history_stage2.history["val_loss"]))

OUTPUT_DIR   = "./performance_plots"   # all performance plots written here

plot_training_curves(history_dict2, out_path=os.path.join(OUTPUT_DIR, "training_curves.png"))

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_11 (Sequential)      │ (None, 64, 64, 8)      │        76,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_7 (Sequential)       │ (None, 8)              │     2,726,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,802,288 (10.69 MB)

 Trainable params: 2,723,752 (10.39 MB)

 Non-trainable params: 78,536 (306.78 KB)

Starting Stage 1: Training Classification Head Only...
Epoch 1/20


KeyboardInterrupt: 